In [ ]:
!pip install sktime

In [ ]:
!pip install optuna

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gc
import optuna
import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.linear_model import RidgeClassifierCV, RidgeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.utils import resample
from sktime.transformations.panel.rocket import MiniRocketMultivariate

# ==========================================
# 1. CARREGAMENTO DOS DADOS
# ==========================================
print("1. Carregando dados do PAMAP2 (v2)...")
data_path = "/content/drive/MyDrive/2025/Estudos/Datasets/pamap2/pamap2_v2.npz"
dados = np.load(data_path)

X_train = np.transpose(dados['X_train'], (0, 2, 1))
X_val   = np.transpose(dados['X_val'], (0, 2, 1))
X_test  = np.transpose(dados['X_test'], (0, 2, 1))
y_train = dados['y_train']
y_val   = dados['y_val']
y_test  = dados['y_test']

del dados
gc.collect()

# ==========================================
# 2. FUNÇÕES BASE (MINIROCKET E FE)
# ==========================================
def extract_minirocket(X_tr, X_v, X_te):
    print("  Extraindo MiniRocket...")
    minirocket = MiniRocketMultivariate()
    minirocket.fit(X_tr)

    def transform_batches(model, X, batch_size=2000):
        features = []
        for i in range(0, X.shape[0], batch_size):
            fim = min(i + batch_size, X.shape[0])
            features.append(model.transform(X[i:fim]).astype(np.float32))
        return np.vstack(features)

    return transform_batches(minirocket, X_tr), transform_batches(minirocket, X_v), transform_batches(minirocket, X_te)

def extract_statistical_features(X):
    # X shape: (N, Canais, Time)
    print("  Extraindo Feature Engineering (Estatísticas: Média, Std, Min, Max, Skew, Kurtosis)...")
    mean = np.mean(X, axis=2)
    std = np.std(X, axis=2)
    max_val = np.max(X, axis=2)
    min_val = np.min(X, axis=2)

    # Assimetria e Curtose (tratando possíveis NaNs gerados por divisões por zero em janelas constantes)
    skewness = skew(X, axis=2, nan_policy='omit')
    kurt = kurtosis(X, axis=2, nan_policy='omit')
    skewness = np.nan_to_num(skewness, nan=0.0)
    kurt = np.nan_to_num(kurt, nan=0.0)

    return np.hstack((mean, std, max_val, min_val, skewness, kurt)).astype(np.float32)

# ==========================================
# 3. FUNÇÃO DO OPTUNA
# ==========================================
def optimize_ridge(X_tr, y_tr, X_v, y_v):
    print("  Otimizando hiperparâmetros com Optuna (F1-Score no Val)...")
    def objective(trial):
        alpha = trial.suggest_float('alpha', 1e-3, 1e3, log=True)
        solver = trial.suggest_categorical('solver', ["auto", "svd", "cholesky", "lsqr", "sparse_cg"])
        clf = make_pipeline(StandardScaler(), RidgeClassifier(alpha=alpha, solver=solver))
        clf.fit(X_tr, y_tr)
        preds_v = clf.predict(X_v)
        return f1_score(y_v, preds_v, average='macro')

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=10) # Número reduzido para demonstração rápida
    return study.best_params

# Pré-computar as features base para economizar tempo
print("\nPré-computando features...")
X_tr_mr, X_v_mr, X_te_mr = extract_minirocket(X_train, X_val, X_test)
X_tr_fe, X_v_fe, X_te_fe = extract_statistical_features(X_train), extract_statistical_features(X_val), extract_statistical_features(X_test)

# Regra de Ouro: Normalizar as features estatísticas manuais antes de qualquer concatenação
print("  Normalizando as features estatísticas (StandardScaler ajustado no Treino)...")
scaler_fe = StandardScaler()
X_tr_fe = scaler_fe.fit_transform(X_tr_fe)
X_v_fe  = scaler_fe.transform(X_v_fe)
X_te_fe = scaler_fe.transform(X_te_fe)

# ==========================================
# 4. EXECUÇÃO DOS CENÁRIOS
# ==========================================
cenarios = [
    {"nome": "1. Sem FE, Sem Optuna", "usa_fe": False, "usa_optuna": False},
    {"nome": "2. Sem FE, Com Optuna", "usa_fe": False, "usa_optuna": True},
    {"nome": "3. Com FE, Sem Optuna", "usa_fe": True,  "usa_optuna": False},
    {"nome": "4. Com FE, Com Optuna", "usa_fe": True,  "usa_optuna": True},
]

resultados = []
fig_cm, axes_cm = plt.subplots(2, 2, figsize=(16, 14))
axes_cm = axes_cm.ravel()

for idx, c in enumerate(cenarios):
    print(f"\n{'='*50}")
    print(f"Executando: {c['nome']}")
    print(f"{'='*50}")

    # Montar o dataset de acordo com o cenário
    if c['usa_fe']:
        # Cenários 3 e 4: Concatena MiniRocket com Estatísticas Manuais (já escalonadas)
        X_tr_final = np.hstack((X_tr_mr, X_tr_fe))
        X_v_final  = np.hstack((X_v_mr, X_v_fe))
        X_te_final = np.hstack((X_te_mr, X_te_fe))
    else:
        # Cenários 1 e 2: Apenas MiniRocket
        X_tr_final = X_tr_mr
        X_v_final  = X_v_mr
        X_te_final = X_te_mr

    # Treinamento e Otimização
    if c['usa_optuna']:
        best_params = optimize_ridge(X_tr_final, y_train, X_v_final, y_val)
        clf = make_pipeline(StandardScaler(), RidgeClassifier(alpha=best_params['alpha'], solver=best_params['solver']))
        clf.fit(X_tr_final, y_train)
        print(f"Melhores Parâmetros Optuna: {best_params}")
    else:
        clf = make_pipeline(StandardScaler(), RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
        clf.fit(X_tr_final, y_train)
        # Acessando o alpha selecionado dentro do pipeline
        best_alpha = clf.named_steps['ridgeclassifiercv'].alpha_
        print(f"Melhor Alpha (CV): {best_alpha:.4f}")

    # Avaliação no Teste
    preds_test = clf.predict(X_te_final)
    acc_test = accuracy_score(y_test, preds_test)
    f1_test = f1_score(y_test, preds_test, average='macro')

    print(f"Acurácia Teste: {acc_test:.4f}")
    print(f"F1-Score (Macro): {f1_test:.4f}")

    print(f"\nRelatório de Classificação - {c['nome']}:")
    print(classification_report(y_test, preds_test))

    resultados.append({
        'Cenário': c['nome'],
        'Acurácia (Mean)': acc_test,
        'F1-Score (Macro Mean)': f1_test
    })

    # Matriz de Confusão para o cenário atual
    ConfusionMatrixDisplay.from_predictions(y_test, preds_test, cmap='Blues', ax=axes_cm[idx], normalize='true')
    axes_cm[idx].set_title(f"Matriz de Confusão - {c['nome']}")

# Exibir todas as matrizes de confusão juntas
plt.tight_layout()
plt.show()

# ==========================================
# 5. RESUMO FINAL E COMPARAÇÃO
# ==========================================
print("\n================ RESUMO FINAL ================")
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

# Gráfico de barras (Histograma comparativo)
df_resultados.set_index('Cenário').plot(kind='bar', figsize=(12, 6), colormap='viridis')
plt.title('Comparação de Acurácia e F1-Score entre os Cenários')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
